# 05 Context Pruning Experiment (Colab)

기존 generation 결과를 덮어쓰지 않고 `experiments/context_pruning_experiment.py`를 Colab에서 실행하기 위한 노트북입니다.

흐름: GPU 확인 → 브랜치 clone/pull → 설치 → Drive mount → 입력 파일 복사 → context-only dry run → generation 실험 → 결과 Drive 저장

In [1]:
from pathlib import Path

REPO_URL = 'https://github.com/beomsookim1020/chatbot.git'
BRANCH = 'colab-generation'
PROJECT_DIR = Path('/content/chatbot')

DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_inputs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_outputs')

PREDICTION_REL = Path('outputs/predictions/best_variant_predictions.jsonl')
EVAL_REL = Path('data/eval/representative_wrong_30_eval_batch_format.csv')
CHUNK_REL = Path('indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl')
SOURCE_STORE_REL = Path('data/source_store_v2_690.jsonl')

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_NEW_TOKENS = 384
RUN_LIMIT = 0  # 0이면 eval CSV 전체. 현재 기본 eval은 wrong 30개입니다.

# 비워두면 A/B/C/D 전체 variant를 실행합니다.
RUN_VARIANTS = []
# 예: RUN_VARIANTS = ['A_current_context_baseline', 'D_hierarchical_context_with_evidence_ranking']

RUN_CONTEXT_ONLY_DRY_RUN = True
CONTEXT_ONLY_DRY_RUN_LIMIT = 2
RUN_GENERATION = True

LOCAL_OUTPUT_ROOT = PROJECT_DIR / 'outputs/context_experiments'
DRIVE_EXPERIMENT_ROOT = DRIVE_OUTPUT_ROOT / 'context_experiments'

print('branch:', BRANCH)
print('model:', MODEL_NAME)
print('eval:', EVAL_REL)
print('predictions:', PREDICTION_REL)
print('chunks:', CHUNK_REL)
print('source_store:', SOURCE_STORE_REL)

branch: colab-generation
model: Qwen/Qwen2.5-3B-Instruct
eval: data/eval/representative_wrong_30_eval_batch_format.csv
predictions: outputs/predictions/best_variant_predictions.jsonl
chunks: indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl
source_store: data/source_store_v2_690.jsonl


## 1. GPU 확인

In [2]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Colab GPU가 켜져 있지 않습니다. Runtime > Change runtime type에서 GPU를 선택하세요.')

!nvidia-smi

torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA L4
Wed May 27 09:05:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   34C    P8             13W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                

## 2. colab-generation 브랜치 clone/pull

In [3]:
import os
import subprocess

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())
subprocess.run(['git', 'status', '--short'], check=True)

cwd: /content/chatbot


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

## 3. 설치

In [4]:
%pip install -q -r requirements.txt
%pip install -q transformers accelerate sentencepiece protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.5/174.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 94.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 123.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 111.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 112.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 95.6 MB/s eta 0:00:00:00:0100:01


## 4. Google Drive mount

In [5]:
from google.colab import drive

drive.mount('/content/drive')
print('Drive input root:', DRIVE_INPUT_ROOT)
print('Drive output root:', DRIVE_OUTPUT_ROOT)

Mounted at /content/drive
Drive input root: /content/drive/MyDrive/chatbot_colab_inputs
Drive output root: /content/drive/MyDrive/chatbot_colab_outputs


## 5. 입력 파일 경로 검증 및 Colab 로컬 복사

In [6]:
import shutil

required_inputs = [
    ('eval', DRIVE_INPUT_ROOT / EVAL_REL),
    ('predictions', DRIVE_INPUT_ROOT / PREDICTION_REL),
    ('chunks', DRIVE_INPUT_ROOT / CHUNK_REL),
    ('source_store', DRIVE_INPUT_ROOT / SOURCE_STORE_REL),
]
missing = [(name, path) for name, path in required_inputs if not path.exists()]
if missing:
    detail = '\n'.join(f'- {name}: {path}' for name, path in missing)
    raise FileNotFoundError('Drive 입력 파일이 없습니다. 경로를 먼저 확인하세요.\n' + detail)

copy_pairs = [
    (DRIVE_INPUT_ROOT / EVAL_REL, PROJECT_DIR / EVAL_REL),
    (DRIVE_INPUT_ROOT / PREDICTION_REL, PROJECT_DIR / PREDICTION_REL),
    (DRIVE_INPUT_ROOT / CHUNK_REL, PROJECT_DIR / CHUNK_REL),
    (DRIVE_INPUT_ROOT / SOURCE_STORE_REL, PROJECT_DIR / SOURCE_STORE_REL),
]
for src, dst in copy_pairs:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'copied: {src} -> {dst} ({dst.stat().st_size:,} bytes)')

script_path = PROJECT_DIR / 'experiments/context_pruning_experiment.py'
if not script_path.exists():
    raise FileNotFoundError(f'실험 스크립트가 없습니다. colab-generation 브랜치에 새 파일이 반영됐는지 확인하세요: {script_path}')
print('script:', script_path)

copied: /content/drive/MyDrive/chatbot_colab_inputs/data/eval/representative_wrong_30_eval_batch_format.csv -> /content/chatbot/data/eval/representative_wrong_30_eval_batch_format.csv (20,087 bytes)
copied: /content/drive/MyDrive/chatbot_colab_inputs/outputs/predictions/best_variant_predictions.jsonl -> /content/chatbot/outputs/predictions/best_variant_predictions.jsonl
copied: /content/drive/MyDrive/chatbot_colab_inputs/indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl -> /content/chatbot/indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl (368,289,817 bytes)
copied: /content/drive/MyDrive/chatbot_colab_inputs/data/source_store_v2_690.jsonl -> /content/chatbot/data/source_store_v2_690.jsonl (553,876,319 bytes)
script: /content/chatbot/experiments/context_pruning_experiment.py


## 6. 실행 함수

In [7]:
import sys
from datetime import datetime

CREATED_OUTPUT_DIRS = []

def list_context_outputs():
    LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    return {path.resolve() for path in LOCAL_OUTPUT_ROOT.glob('context_pruning_*') if path.is_dir()}

def run_context_experiment(*, context_only: bool, limit: int, run_name: str):
    before = list_context_outputs()
    cmd = [
        sys.executable,
        str(PROJECT_DIR / 'experiments/context_pruning_experiment.py'),
        '--predictions', str(PREDICTION_REL),
        '--eval-csv', str(EVAL_REL),
        '--chunks', str(CHUNK_REL),
        '--source-store', str(SOURCE_STORE_REL),
        '--output-root', 'outputs/context_experiments',
        '--run-name', run_name,
        '--model-name', MODEL_NAME,
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--limit', str(limit),
    ]
    if context_only:
        cmd.append('--context-only')
    for variant in RUN_VARIANTS:
        cmd.extend(['--variant', variant])
    print('Running command:')
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
    after = list_context_outputs()
    created = sorted(after - before, key=lambda path: path.stat().st_mtime)
    if not created:
        raise RuntimeError('새 output directory를 찾지 못했습니다.')
    CREATED_OUTPUT_DIRS.extend(created)
    print('created output:', created[-1])
    return created[-1]

## 7. Context-only dry run

모델을 로드하지 않고 context 구성만 먼저 확인합니다.

In [8]:
if RUN_CONTEXT_ONLY_DRY_RUN:
    dry_output_dir = run_context_experiment(
        context_only=True,
        limit=CONTEXT_ONLY_DRY_RUN_LIMIT,
        run_name='context_pruning_dry_run',
    )
else:
    dry_output_dir = None
    print('Context-only dry run skipped.')

Running command:
/usr/bin/python3 /content/chatbot/experiments/context_pruning_experiment.py --predictions outputs/predictions/best_variant_predictions.jsonl --eval-csv data/eval/representative_wrong_30_eval_batch_format.csv --chunks indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl --source-store data/source_store_v2_690.jsonl --output-root outputs/context_experiments --run-name context_pruning_dry_run --model-name Qwen/Qwen2.5-3B-Instruct --max-new-tokens 384 --limit 2 --context-only
created output: /content/chatbot/outputs/context_experiments/context_pruning_dry_run_20260527_090630


## 8. Generation 실험 실행

In [9]:
if RUN_GENERATION:
    generation_output_dir = run_context_experiment(
        context_only=False,
        limit=RUN_LIMIT,
        run_name='context_pruning',
    )
else:
    generation_output_dir = None
    print('Generation skipped.')

Running command:
/usr/bin/python3 /content/chatbot/experiments/context_pruning_experiment.py --predictions outputs/predictions/best_variant_predictions.jsonl --eval-csv data/eval/representative_wrong_30_eval_batch_format.csv --chunks indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl --source-store data/source_store_v2_690.jsonl --output-root outputs/context_experiments --run-name context_pruning --model-name Qwen/Qwen2.5-3B-Instruct --max-new-tokens 384 --limit 0


KeyboardInterrupt: 

## 9. 결과 확인

In [ ]:
import pandas as pd

latest_output_dir = generation_output_dir or dry_output_dir
if latest_output_dir is None:
    raise RuntimeError('확인할 output directory가 없습니다.')

metrics_path = latest_output_dir / 'context_pruning_metrics.csv'
review_path = latest_output_dir / 'context_pruning_review.csv'
summary_path = latest_output_dir / 'context_pruning_summary.md'
results_path = latest_output_dir / 'context_pruning_results.jsonl'

print('results:', results_path)
print('review:', review_path)
print('metrics:', metrics_path)
print('summary:', summary_path)

display(pd.read_csv(metrics_path))
display(pd.read_csv(review_path).head(10))

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 10. Drive로 결과 저장

In [ ]:
DRIVE_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

copied_dirs = []
for local_dir in CREATED_OUTPUT_DIRS:
    dst = DRIVE_EXPERIMENT_ROOT / local_dir.name
    if dst.exists():
        raise FileExistsError(f'Drive 결과 폴더가 이미 있습니다. 덮어쓰지 않습니다: {dst}')
    shutil.copytree(local_dir, dst)
    copied_dirs.append(dst)
    print('copied output to Drive:', dst)

if not copied_dirs:
    print('복사할 새 결과 폴더가 없습니다.')
else:
    print('Drive output dirs:')
    for path in copied_dirs:
        print('-', path)

## 사람이 볼 파일

- `context_pruning_review.csv`: 사람이 `manual_correct`, `failure_type`, `review_note`를 채우는 파일
- `context_pruning_metrics.csv`: variant별 context 길이, gold signal proxy, source_store 사용률 비교
- `context_pruning_summary.md`: 기존 구현 여부와 실패 사례 요약
- `context_pruning_results.jsonl`: 전체 generation 상세 결과와 `used_context` 포함